Here’s a PyTorch snippet to compute candidate headings from your LOS velocity equation and embed them into node features:

✅ Physics-Informed Heading Calculation in PyTorch

In [13]:

import torch
import math

# Example input: positions and LOS velocity
# x_t, y_t: target positions
# x_s, y_s, z_s: sensor position
# v_los: LOS velocity for each target
# v_assumed: assumed speed for heading calculation

def compute_candidate_headings(target_positions, v_los, sensor_position, v_assumed):
    """
    Compute two candidate headings for each target based on LOS velocity equation.
    
    Args:
        target_positions: Tensor [num_targets, 2] -> (x_t, y_t)
        v_los: Tensor [num_targets] -> LOS velocity
        sensor_position: Tensor [3] -> (x_s, y_s, z_s)
        v_assumed: float -> assumed target speed
        
    Returns:
        headings: Tensor [num_targets, 2] -> two candidate headings per target
    """
    x_t, y_t = target_positions[:, 0], target_positions[:, 1]
    x_s, y_s, z_s = sensor_position

    # Compute range from sensor to target
    dx = x_t - x_s
    dy = y_t - y_s
    range_3d = torch.sqrt(dx**2 + dy**2 + z_s**2)

    # Compute normalized projection factor
    factor = v_los * range_3d / v_assumed

    # Clip factor to [-r, r] to avoid invalid acos
    r = torch.sqrt(dx**2 + dy**2)
    
    
    factor = torch.clamp(factor / r, -1.0, 1.0)

    # Compute candidate headings using inverse cosine
    # Equation rearranged: cos(theta) = factor * dx / r, sin(theta) = factor * dy / r
    # Simplify by assuming heading angle satisfies:
    # (dx * cos(theta) + dy * sin(theta)) / r = factor
    # So theta = atan2(dy, dx) ± acos(factor)
    base_angle = torch.atan2(dy, dx)
    delta_angle = torch.acos(factor)

    heading1 = base_angle + delta_angle
    heading2 = base_angle - delta_angle

    return torch.stack([heading1, heading2], dim=1)

# Example usage
num_targets = 5
target_positions = torch.tensor([[1000., 200.],
                                  [1200., 250.],
                                  [1300., 300.],
                                  [1400., 350.],
                                  [1500., 400.]])
#v_los = torch.tensor([10., 12., 11., 9., 8.])  # LOS velocities
v_los = torch.tensor([1.0, 1.2, 1.1, 0.9, 0.8])  # LOS velocities

sensor_position = torch.tensor([0., 0., 10000.])  # Sensor high above
v_assumed = 20.0  # Assume 20 m/s speed

headings = compute_candidate_headings(target_positions, v_los, sensor_position, v_assumed)
print("Candidate headings (degrees):\n", torch.rad2deg(headings))


Candidate headings (degrees):
 tensor([[ 71.7830, -49.1631],
        [ 72.2202, -48.6836],
        [ 78.4189, -52.4297],
        [ 85.6717, -57.5992],
        [ 89.8169, -59.9541]])
